<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB19_Case_Study_CLIWOC_Nationality_from_Ship_Routes_ES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NB19 · Clase 19 — Caso de estudio: cuadernos de bitácora históricos CLIWOC, clasificando la nacionalidad a partir de las rutas**

## Bloque 4: Proyectos — Casos de estudio (continuación)

`NB18` usó datos meteorológicos reales del siglo XXI. Este caso de estudio se remonta mucho más atrás: cuadernos de bitácora reales de barcos del siglo XVIII, digitalizados para investigación climática, usados aquí para plantear un tipo de pregunta distinto — ¿puede la **ruta y el momento** de un barco, por sí solos, revelar para qué nación navegaba?

**El problema real**: entre mediados y finales del siglo XVIII, británicos, neerlandeses, españoles y franceses mantenían cada uno redes comerciales marítimas consolidadas, ligadas a sus posesiones coloniales — las rutas transatlánticas y del galeón de Manila de España, la ruta del Cabo de la VOC neerlandesa hacia las Indias Orientales, el comercio atlántico y de la Compañía Británica de las Indias Orientales, y el comercio caribeño y del océano Índico francés. Si esas redes eran reales y geográficamente distintas, un modelo debería poder recuperar "de qué nación" a partir únicamente de *dónde* y *cuándo* estaba un barco — `una prueba genuina de si la geografía comercial histórica real es aprendible a partir de datos, y no una etiqueta arbitraria de clase`.

Descargamos el dataset real en directo desde Kaggle, así que las filas exactas y los años disponibles pueden variar ligeramente según la versión actual — este notebook está escrito deliberadamente para **descubrir** la estructura real de los datos y su rango temporal, en lugar de asumir cifras fijas, exactamente el hábito sobre el que se construyó el flujo "cargar → inspeccionar" de `NB02`.

Siguiendo la estructura corregida de `NB18`, esta clase vuelve a tener **dos partes**: la **Parte A** (Secciones 8–9) valida un enfoque de modelado con una división aleatoria estándar; la **Parte B** (Sección 10) es la prueba real — entrenando solo con el ~80% más antiguo de los años disponibles y prediciendo la nacionalidad para los **años posteriores que el modelo nunca ha visto**, comprobando si estos patrones de rutas comerciales se mantuvieron estables en el tiempo, o cambiaron.

### Objetivos de aprendizaje

Al final de esta clase, el alumnado será capaz de:
- Explicar por qué "ruta → nacionalidad" es una pregunta real, con base histórica, y no una etiqueta arbitraria.
- Descargar un dataset real de Kaggle usando credenciales seguras, válidas solo para la sesión.
- Trabajar con un dataset histórico real y desordenado — incluyendo descubrir su estructura real y el balance de clases, en lugar de asumirlos de antemano.
- Visualizar datos geográficos separados por clase sobre un mapa real y usarlo para comprobar la coherencia de una premisa de modelado antes de entrenar nada.
- Gestionar el desequilibrio de clases con una referencia ingenua (baseline) y `class_weight="balanced"`.
- Diseñar una reserva temporal (holdout) que encaje con la pregunta real planteada (generalizar entre años, sin que se le haya dado ya la "respuesta" que ya vio).

### Agenda (clase de 2 horas)

| # | Segmento de la clase | Duración aprox. | Tipo |
|---|---------------------|:---:|:---:|
| 1 | Repaso, hoja de ruta de hoy | 5 min | Teoría |
| 2 | Qué es CLIWOC, y por qué "ruta → nacionalidad" es una pregunta real | 10 min | Teoría |
| 3 | Configurar el acceso a la API de Kaggle | 5 min | Práctica |
| 4 | Descargar el dataset real | 5 min | Práctica |
| 5 | Explorar los datos: columnas, nacionalidades, rutas en un mapa real | 15 min | Práctica |
| 6 | Preparar las características y afrontar el desequilibrio real de clases | 15 min | Teoría + Práctica |
| 7 | Aplicar el marco de decisión de `NB17` | 5 min | Teoría + Práctica |
| 8 | Parte A: entrenar y comparar modelos (validación de la metodología) | 15 min | Práctica |
| 9 | Parte A: evaluación | 20 min | Práctica |
| 10 | Parte B: una prueba genuina — predecir años reservados | 10 min | Práctica |
| 11 | Interpretar la Parte B y conectarla con la historia real | 10 min | Práctica |
| 12 | Resumen, tarea, próxima clase | 5 min | Teoría |

> Los tiempos son una guía aproximada, no un guion estricto — no hay descansos programados. Si cubrimos todo con tiempo de sobra, la clase termina antes; eso puede pasar y está bien.


---

## 1. Repaso: dónde estamos

- **`NB18`**: datos meteorológicos reales de ECMWF, una predicción genuina sobre un año reservado, y la lección de que una prueba de generalización justa tiene que reservar la variable *correcta* (un año, no una estación, cuando el objetivo es estacional).
- **`NB19`** (hoy): un tipo distinto de datos reales — históricos, escasos, desequilibrados — y la misma disciplina aplicada a una reserva temporal que encaja con la pregunta real de *este* problema.

---

## 2. Qué es CLIWOC, y por qué "ruta → nacionalidad" es una pregunta real

**[CLIWOC](https://en.wikipedia.org/wiki/CLIWOC)** (Climatological Database for the World's Oceans) fue un proyecto de investigación real que convirtió cuadernos de bitácora históricos de barcos — británicos, neerlandeses, franceses y españoles, 1750–1850 — en una base de datos digital estandarizada, originalmente para reconstruir patrones climáticos y de viento históricos a partir de siglos de observaciones diarias al mediodía. Eso significa que cada fila de este dataset es una **entrada real que escribió un oficial de barco real** en el mar, hace hasta 275 años.

La pregunta de hoy usa los mismos datos con otro propósito: cada una de las cuatro naciones mantenía redes comerciales reales y distintas, moldeadas por sus territorios y monopolios coloniales — las rutas transatlánticas y del galeón del Pacífico de España, la ruta del Cabo de Buena Esperanza de la Compañía neerlandesa de las Indias Orientales hacia Indonesia, el comercio atlántico y del océano Índico británico, y el comercio caribeño y del océano Índico francés. Si esas redes eran geográficamente reales y distintas (que es lo que dice la historia marítima real), un modelo entrenado solo con **dónde** y **cuándo** se registró una entrada del cuaderno debería poder recuperar **de qué nación** era — `geografía histórica genuina, aprendible a partir de datos, no una etiqueta arbitraria inventada para un ejercicio de clase`.

---

## 3. Configurar el acceso a la API de Kaggle

La base de datos completa de CLIWOC — más de 287.000 entradas reales de cuadernos de bitácora, 141 columnas, entre 1750 y 1850 — está publicada en Kaggle. Descargarla requiere una cuenta gratuita de Kaggle y una clave de API:

1. Ve a [kaggle.com/settings](https://www.kaggle.com/settings), baja hasta **API**, y pulsa **Create New Token** — esto descarga un fichero `kaggle.json` a tu ordenador.
2. Ejecuta la celda de abajo; mostrará un **selector de archivos** — elige el `kaggle.json` que acabas de descargar.

Los bytes del fichero van directamente al almacenamiento privado de este entorno de Colab y nunca se escriben en el código guardado ni en la salida de este notebook — una credencial real nunca debería quedar como texto plano y confirmado (commit) en una celda, que es exactamente cómo este repositorio filtró una clave real de Kaggle al principio de la historia de este curso.

In [ ]:
%pip install -q kaggle

import os
from google.colab import files

print("Select your kaggle.json file:")
uploaded = files.upload()

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
kaggle_json_path = os.path.expanduser("~/.kaggle/kaggle.json")
with open(kaggle_json_path, "wb") as f:
    f.write(next(iter(uploaded.values())))
os.chmod(kaggle_json_path, 0o600)

print("Kaggle credentials saved for this session (not stored anywhere in this notebook).")

---

## 4. Descargar el dataset real

In [ ]:
!kaggle datasets download -d cwiloc/climate-data-from-ocean-ships --force --unzip

import os

csv_files = [f for f in os.listdir(".") if f.lower().endswith(".csv")]
print("CSV files in the downloaded archive:", csv_files)

---

## 5. Explorar los datos: columnas, nacionalidades, rutas en un mapa real

Las preguntas de partida de `NB02`, esta vez sobre datos reales del siglo XVIII. El fichero principal del archivo normalmente es `CLIWOC15.csv` — cárgalo y mira primero sus columnas reales, exactamente como el hábito "cargar → inspeccionar" de `NB02`, ya que con 180 columnas reales no tiene sentido adivinar:

In [ ]:
import pandas as pd

raw = pd.read_csv("CLIWOC15.csv", low_memory=False)
print(raw.shape)
raw.columns.tolist()

Las columnas relevantes para la pregunta de hoy sobre las cuatro naciones y sus rutas: `Lon3`/`Lat3` (una posición decimal depurada, entre varias representaciones de posición de este dataset), `Year`, y `Nationality`. Quédate solo con las filas donde las cuatro estén presentes, y restringe a las cuatro armadas bien representadas de las que trata esta clase — comparando los valores sin distinguir mayúsculas/minúsculas, ya que todavía no hemos confirmado su formato exacto en esta versión:

In [ ]:
print(raw["Nationality"].value_counts())

Filtra las filas reales y completas para nuestras cuatro naciones objetivo, y renombra las columnas de posición a `longitude`/`latitude` simples, para que el resto del notebook no necesite conocer los nombres de columna originales:

In [ ]:
target_nations = ["BRITISH", "DUTCH", "SPANISH", "FRENCH"]
nat_upper = raw["Nationality"].astype(str).str.upper().str.strip()

cliwoc = raw.loc[nat_upper.isin(target_nations), ["Lon3", "Lat3", "Year", "Nationality"]].copy()
cliwoc["Nationality"] = nat_upper[nat_upper.isin(target_nations)]
cliwoc = cliwoc.rename(columns={"Lon3": "longitude", "Lat3": "latitude"}).dropna()

print(cliwoc.shape)
cliwoc.head()

Ahora el resumen real, descubierto — sin suposiciones, solo lo que esta descarga realmente contiene:

In [ ]:
print(cliwoc["Nationality"].value_counts())
print()
print("Year range:", cliwoc["Year"].min(), "-", cliwoc["Year"].max())

**Lee tu propia salida**: ¿están las cuatro clases cerca de estar equilibradas, o hay una nación notablemente más rara que las demás? La cobertura real del archivo (cuántos cuadernos de bitácora de cada nación sobrevivieron y fueron digitalizados) `rara vez produce un dataset perfectamente equilibrado — un desequilibrio del mundo real que merece la pena arrastrar`, en el mismo espíritu que `NB13` señaló para sus propias etiquetas reales, autoderivadas.

No solo los totales generales — comprueba si los cuadernos de bitácora de cada nación abarcan los *mismos* años, o si los registros de algunas naciones se digitalizaron para un periodo más estrecho que los de otras. Esto no es curiosidad ociosa: afecta directamente al diseño de los años reservados de la Parte B más adelante.

In [ ]:
print(cliwoc.groupby("Nationality")["Year"].agg(["min", "max", "count"]))

**Lee tu propia salida**: si los cuadernos de bitácora de una o más naciones terminan bastante antes que el último año del dataset, un corte sencillo del "último 20% de todos los años" acabaría probando casi por completo sobre la nación cuyos registros llegan más lejos — un artefacto de una cobertura de digitalización desigual, no una prueba genuina de la estabilidad de las rutas. La Sección 10 elige su corte teniendo esto en cuenta, restringiendo los años candidatos a aquellos en los que todas las naciones todavía tienen entradas reales.

Antes de entrenar nada, comprueba visualmente la premisa real: ¿traza cada nación un patrón geográficamente distinto en sus datos reales de cuaderno de bitácora?

In [ ]:
%pip install -q cartopy

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

sample = cliwoc.sample(min(5000, len(cliwoc)), random_state=42)
colors = {"BRITISH": "tab:blue", "DUTCH": "tab:orange", "SPANISH": "tab:green", "FRENCH": "tab:red"}

fig = plt.figure(figsize=(12, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines(resolution="110m")
ax.add_feature(cfeature.BORDERS, linestyle=":")
ax.add_feature(cfeature.LAND, facecolor="whitesmoke")

for nation, color in colors.items():
    subset = sample[sample["Nationality"] == nation]
    ax.scatter(subset["longitude"], subset["latitude"], s=4, alpha=0.5,
               label=nation, color=color, transform=ccrs.PlateCarree())

ax.legend(markerscale=3, loc="lower left")
year_lo, year_hi = int(cliwoc["Year"].min()), int(cliwoc["Year"].max())
ax.set_title(f"Real CLIWOC logbook positions by nationality, {year_lo}-{year_hi} ({len(sample):,}-entry sample)")
plt.show()

**Lee tu propio mapa**: ¿puedes ver las rutas transatlánticas/del Pacífico españolas, el corredor neerlandés del Cabo de Buena Esperanza hacia Indonesia, la presencia británica en el Atlántico y el océano Índico? Si estos cuatro colores se separan en regiones visualmente distintas, `eso es evidencia real y directa de que la premisa "la ruta revela la nacionalidad" se sostiene` *antes* de entrenar ningún modelo — el propio mapa está haciendo un análisis exploratorio de datos genuino, al estilo `NB02`, sobre una pregunta histórica real.

---

## 6. Preparar las características y afrontar el desequilibrio real de clases

Características: `latitude`, `longitude` (dónde) y `Year` (cuándo). Objetivo: `Nationality`. (`CLIWOC15.csv` no lleva de forma fiable una columna `Month` limpia en todas las entradas del mismo modo que `Year`, así que mantenemos el conjunto de características en lo que hemos confirmado que está completo.)

In [ ]:
feature_cols = ["latitude", "longitude", "Year"]
X = cliwoc[feature_cols]
y = cliwoc["Nationality"]

print(y.value_counts(normalize=True).round(3))

Ver el desequilibrio como un diagrama de barras facilita juzgarlo de un vistazo:

In [ ]:
counts = y.value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(counts.index, counts.values, color="steelblue")
ax.set_ylabel("Number of entries")
ax.set_title("Class balance across the four nations")
plt.show()


**Lee la salida de la celda de arriba**: sea cual sea la nación que resulte más rara, un modelo que la ignore por completo aún podría obtener una alta accuracy global — el aviso original sobre el desequilibrio de `NB07`, ahora con cifras reales detrás. Usaremos dos herramientas concretas contra esto: una **referencia ingenua (baseline)** para saber qué puntuación obtendría realmente "hacer trampa ignorando la clase minoritaria", y la opción `class_weight="balanced"` de scikit-learn, que aumenta el peso de los errores en la clase minoritaria durante el entrenamiento en lugar de tratar todos los errores por igual.

---

## 7. Aplicar el marco de decisión de `NB17`

1. **¿Etiquetas?** Sí — `Nationality` es real y se conoce para cada entrada.
2. **¿Forma de los datos?** Tabular — tres características numéricas simples por fila.
3. **¿Volumen de datos?** Mira `X.shape[0]` de arriba — probablemente decenas de miles de filas o más, cómodamente en el rango donde tanto el ML clásico como una red neuronal podrían funcionar; el marco de `NB17` no favorece claramente a uno de los dos aquí, como sí hacía con los 308 datos del yate de `NB10`.

Dado que esta vez el marco no empuja con fuerza en ninguna dirección, usaremos un ensemble clásico (rápido, interpretable, fácil de ponderar para el desequilibrio) — pero ten en cuenta que este es un caso en el que probar un pequeño MLP (al estilo `NB11`) como tarea es una pregunta genuinamente abierta, no una conclusión cantada.

---

## 8. Parte A: entrenar y comparar modelos (validación de la metodología)

Primero una división aleatoria y estratificada — el mismo tipo de paso de "validar el enfoque" que la Parte A de `NB18`:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)

Compara la referencia ingenua con un Random Forest ponderado por clase:

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_scaled, y_train)
print("Naive baseline (always predict the majority class) accuracy:", round(baseline.score(X_test_scaled, y_test), 3))

rf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
rf.fit(X_train_scaled, y_train)
print("Random Forest accuracy:", round(rf.score(X_test_scaled, y_test), 3))

---

## 9. Parte A: evaluación

La accuracy global esconde cómo le va a cada nación individualmente — un informe completo importa aquí más de lo habitual, dado el desequilibrio:

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = rf.predict(X_test_scaled)
print(confusion_matrix(y_test, y_pred, labels=rf.classes_))
print()
print(classification_report(y_test, y_pred, labels=rf.classes_))

Una matriz de confusión visual, de la misma forma en que la mostraron `NB07`/`NB11`:

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred, labels=rf.classes_), display_labels=rf.classes_).plot(cmap="Blues", ax=ax, xticks_rotation=45)
plt.show()


**Pruébalo tú mismo**: extrae el recall individual de cada nación del classification report y represéntalo — ¿se ve el desequilibrio directamente en el recall por clase, y no solo en los recuentos brutos de la Sección 6?

In [ ]:
report_dict = classification_report(y_test, y_pred, labels=rf.classes_, output_dict=True)
recalls = {nation: report_dict[nation]["recall"] for nation in rf.classes_}

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(recalls.keys(), recalls.values(), color="darkorange")
ax.set_ylabel("Recall")
ax.set_title("Part A: recall per nation")
ax.set_ylim(0, 1)
plt.show()


**Lee tu propio informe**: ¿es el recall de la nación más rara notablemente más bajo que el de las otras tres, incluso con `class_weight="balanced"`? Eso sería un resultado honesto y esperable, dado lo poca cantidad de datos que existe para ella — una limitación real del archivo subyacente, no un error de modelado que haya que corregir.

---

## 10. Parte B: una prueba genuina — predecir años reservados

La división aleatoria de la Parte A mezcla entradas de todas las épocas tanto en train como en test — una comprobación de metodología justa, pero no una prueba real de si estos patrones de rutas *se mantuvieron a lo largo del tiempo*. La pregunta genuina: entrenar solo con los años más antiguos, y luego predecir la nacionalidad para **años que el modelo nunca ha visto**, exactamente la misma disciplina que usó `NB18` para su año reservado, aplicada a la variable que realmente importa para *esta* pregunta (el periodo temporal, no la estación).

La comprobación de cobertura de la Sección 5 puede que ya haya mostrado que los cuadernos de bitácora de algunas naciones terminan años antes que los de otras. Para que esto siga siendo una prueba justa y no una que colapse silenciosamente en "predecir la nación cuyo archivo llega más lejos", el corte de abajo se elige únicamente entre **años en los que todas las naciones todavía tienen entradas reales** — la intersección de los años cubiertos por cada nación, no el rango de años de todo el dataset.

Un ajuste más, por la misma razón por la que `NB18` descartó su propia característica `year` antes de agrupar varios años de entrenamiento: `Year` era una entrada legítima para la Parte A (train y test procedían del mismo rango), pero todo el sentido de la Parte B es probar con años que el modelo nunca ha visto — pasarle `Year` como un número bruto a un modelo basado en árboles, que luego tiene que puntuar filas con valores de `Year` *más allá* de cualquier umbral por el que alguna vez dividió, es una limitación real de cómo extrapolan los ensembles de árboles, no una prueba justa de si las **rutas** generalizan. La Parte B entrena solo con `latitude`/`longitude`, así que cualquier caída de accuracy refleja la geografía en sí misma, y no que el modelo falle al extrapolar un número para el que nunca se construyó para extrapolar.

Resulta que la cobertura real de CLIWOC aquí abarca **siglos**, no décadas (esta es justo la razón por la que el notebook descubre su propio rango de años en lugar de asumir uno). Probar con "todos los años desde el corte hasta el final del archivo" `compararía silenciosamente la geografía comercial colonial temprana con todo lo ocurrido hasta mediados del siglo XIX` — incluyendo las guerras napoleónicas, la disolución de la VOC neerlandesa en 1799, y las guerras de independencia hispanoamericanas, una reorganización completa del comercio colonial que no tiene nada que ver con si las rutas son estables en el *corto* plazo. Así que la Parte B reserva una **ventana acotada justo después del corte** — comparable en espíritu al único año reservado de `NB18` — no un "resto de la historia" abierto.

Última comprobación: la presencia real de algunas naciones en los cuadernos de bitácora se concentra en ráfagas históricas concretas (una expedición concreta, una década concreta de comercio activo) en lugar de repartirse uniformemente a lo largo de dos siglos. Una nación con solo un puñado de entradas reales en esta ventana concreta no puede sostener una prueba de clasificación justa, por mucho que se ajuste el modelo — eso es un límite genuino de escasez de datos, no algo que se arregle con otra división. Así que la Parte B **descarta automáticamente cualquier nación por debajo de una proporción mínima real de la ventana de test**, y lo indica explícitamente, en lugar de forzar una comparación que un puñado de filas no puede sostener honestamente:

In [ ]:
feature_cols_era = ["latitude", "longitude"]
TEST_WINDOW_YEARS = 10
MIN_TEST_SHARE = 0.05  # a nation needs at least 5% of the test window's real rows

years_per_nation = [set(cliwoc.loc[cliwoc["Nationality"] == nat, "Year"]) for nat in cliwoc["Nationality"].unique()]
common_years = sorted(set.intersection(*years_per_nation))

if len(common_years) < 5:
    raise ValueError(
        "Fewer than 5 years have entries from every nation -- the four nations' "
        "logbook coverage barely overlaps in time, so a fair year-based holdout "
        "isn't possible here. Inspect Section 5's per-nation year ranges above "
        "to see which nation's coverage is the bottleneck."
    )

cutoff_year = common_years[int(len(common_years) * 0.8)]
window_end_year = cutoff_year + TEST_WINDOW_YEARS

train_era = cliwoc[cliwoc["Year"] < cutoff_year]
test_era = cliwoc[(cliwoc["Year"] >= cutoff_year) & (cliwoc["Year"] < window_end_year)]

if len(test_era) == 0:
    raise ValueError(
        f"No entries fall in the {cutoff_year}-{window_end_year - 1} test window -- "
        "try a larger TEST_WINDOW_YEARS, or inspect Section 5's per-nation year "
        "ranges to see where coverage actually thins out."
    )

nation_share = test_era["Nationality"].value_counts(normalize=True)
stable_nations = sorted(nation_share[nation_share >= MIN_TEST_SHARE].index)
dropped_nations = sorted(set(cliwoc["Nationality"].unique()) - set(stable_nations))

if dropped_nations:
    print(f"Dropping {dropped_nations} from Part B: under {MIN_TEST_SHARE:.0%} of the "
          f"{cutoff_year}-{window_end_year - 1} window's real entries -- too sparse in "
          "this specific era for a fair test, even though Part A's full random split "
          "above had enough data for them.")

train_era = train_era[train_era["Nationality"].isin(stable_nations)]
test_era = test_era[test_era["Nationality"].isin(stable_nations)]

X_train_era, y_train_era = train_era[feature_cols_era], train_era["Nationality"]
X_test_era, y_test_era = test_era[feature_cols_era], test_era["Nationality"]

print(f"Held-out cutoff year: {cutoff_year} (chosen from {len(common_years)} years common to all four nations)")
print(f"Part B nations: {stable_nations}")
print(f"Train (years before {cutoff_year}):", X_train_era.shape)
print(f"Test  (years {cutoff_year}-{window_end_year - 1}):", X_test_era.shape)
print(y_test_era.value_counts(normalize=True).round(3))

Entrena un modelo nuevo — esto **no** debe reutilizar el modelo de la Parte A, que ya vio algunas filas de la era reservada en su propia división de entrenamiento:

In [ ]:
scaler_era = StandardScaler()
X_train_era_scaled = scaler_era.fit_transform(X_train_era)
X_test_era_scaled = scaler_era.transform(X_test_era)

rf_era = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
rf_era.fit(X_train_era_scaled, y_train_era)

y_pred_era = rf_era.predict(X_test_era_scaled)
print(classification_report(y_test_era, y_pred_era, labels=rf_era.classes_))

**Pruébalo tú mismo**: no te limites a comparar a ojo dos informes impresos — reúne el recall de cada nación de la Parte A y de la Parte B en una sola tabla, lado a lado.

In [ ]:
report_dict_a = classification_report(y_test, y_pred, labels=rf.classes_, output_dict=True)
report_dict_b = classification_report(y_test_era, y_pred_era, labels=rf_era.classes_, output_dict=True, zero_division=0)

recall_comparison = pd.DataFrame({
    "Part A recall": {nat: report_dict_a[nat]["recall"] for nat in rf.classes_},
    "Part B recall": {nat: report_dict_b[nat]["recall"] for nat in rf_era.classes_},
})
recall_comparison.round(3)


---

## 11. Interpretar la Parte B y conectarla con la historia real

**Si la Sección 10 imprimió una lista de `Part B nations` más corta que las cuatro de la Parte A**, eso en sí mismo es un hallazgo real que merece la pena decir con claridad: la nación que se descartó simplemente no tiene suficientes entradas reales de cuaderno de bitácora en esta ventana concreta para sostener una prueba justa, aunque en total tuviera bastantes datos (la Parte A la entrenó sin problema). Los archivos reales no están distribuidos uniformemente en el tiempo — la presencia documentada de algunas naciones se concentra en décadas concretas, y no se reparte suavemente a lo largo de los siglos.

**Compara este informe con el de la Parte A, para las naciones que quedan.** Una caída significativa sería un hallazgo histórico genuinamente interesante, no un fallo: `sugeriría que las rutas comerciales cambiaron justo alrededor del corte, incluso dentro de la ventana corta y acotada de la Sección 10`. Aquí se espera una caída — la división aleatoria de la Parte A deja que el modelo vea todas las épocas a la vez, mientras que la Parte B genuinamente no; una caída *moderada* (no un colapso casi a cero en todas las clases) es un resultado legítimo, no algo que haya que seguir ajustando para que desaparezca.

Busca el `cutoff_year` impreso en la celda de arriba, y comprueba qué estaba pasando históricamente justo alrededor de esa fecha para las naciones de este dataset. Si tus cifras de recall/precision difieren notablemente **entre naciones** (no solo en conjunto), merece la pena investigarlo nación por nación, no solo como una puntuación agregada — distintas naciones pueden verse alteradas por distintos sucesos reales en los mismos años. Por ejemplo, un corte que caiga alrededor de **1789** se sitúa al comienzo de una década genuinamente turbulenta específicamente para los **neerlandeses**: la Revolución de los Patriotas (años 1780) y la invasión prusiana (1787) desestabilizaron la política de la República Neerlandesa, y la invasión del ejército revolucionario francés en 1795 (que creó la República Bátava) desencadenó la crisis financiera terminal del monopolio comercial de la VOC, que se disolvió formalmente en 1799 — una alteración del comercio colonial neerlandés en concreto, distinta de lo que enfrentaron los barcos británicos o españoles en esos mismos años. Si las predicciones neerlandesas salen notablemente peor que las de las demás naciones en tu propia ejecución, esa historia real es una explicación plausible; si la nación más débil es otra, o tu corte cae en otro punto completamente distinto, busca qué estaba pasando para *esa* nación alrededor de *ese* año, en lugar de reutilizar este ejemplo.

Este es exactamente el uso honesto de una reserva genuina: no se limita a puntuar un modelo, puede **sacar a la luz una pregunta histórica real que merece la pena investigar más** (¿hubo un cambio real, y si lo hubo, en las rutas de qué nación en concreto?) — una hipótesis que este notebook plantea pero no pretende demostrar; haría falta un análisis histórico real, no solo una cifra de accuracy, para confirmarla.

---

## Resumen de la clase

- CLIWOC convierte cuadernos de bitácora de barcos del siglo XVIII en datos reales y estructurados — observaciones reales, desequilibrio de clases real, implicaciones históricas reales detrás de las cifras, descargados en directo desde Kaggle con credenciales seguras y válidas solo para la sesión.
- "La ruta revela la nacionalidad" es una afirmación histórica genuina, comprobada visualmente sobre un mapa real antes de que ningún modelo tocara los datos.
- Una referencia ingenua y `class_weight="balanced"` son dos herramientas concretas y complementarias contra el desequilibrio real de clases — usadas juntas, no como sustituto de leer el informe completo por clase.
- La Parte A (división aleatoria) valida un enfoque de modelado; la Parte B (entrenar con los años más antiguos, probar en una ventana corta y acotada justo después del corte) es la prueba real — ajustando tanto la variable reservada como su alcance temporal a la pregunta real, exactamente como estableció `NB18`.
- Una caída real de accuracy en una reserva genuina no es un fallo que haya que justificar — es una señal legítima que merece la pena conectar con la historia real.

## Para la próxima clase

Otro caso de estudio del Bloque 4, usando datos reales de terreno/elevación — continuando el mismo patrón: un dataset real, una pregunta real, y una reserva genuina (no solo metodológica) allí donde la pregunta lo requiera.

## Tarea / Ideas de práctica

1. Añade `ShipType` (del dataset en bruto, codificado) como una cuarta característica — ¿mejora el recall por clase de la Parte A, especialmente para la nación más rara?
2. Prueba un pequeño MLP (al estilo `NB11`) en esta tarea, según la pregunta abierta de la Sección 7 — ¿supera aquí al Random Forest, dado cuántos más datos tiene este dataset que la mayoría de las otras tareas de clasificación de este curso?
3. Cambia `TEST_WINDOW_YEARS` en la Sección 10 (prueba con 3, y luego con 20) — ¿recupera una ventana más corta más de la accuracy de la Parte A, y colapsa aún más una mucho más larga, tal como hacía la versión original de "resto de la historia" sin límite?
4. Vuelve a calcular la Parte A usando `class_weight=None` (el valor por defecto, sin ponderar) en lugar de `"balanced"` — ¿cuánto cambia el recall de la nación más rara, y sube o baja la accuracy global?
5. Usando el mapa de la Sección 5, elige una nación y describe (en una celda markdown) cómo deberían ser sus rutas comerciales históricas reales — ¿coinciden los datos representados con tu propia expectativa histórica?

> ***Como siempre: un dataset histórico real puede enseñarte tanto sobre historia como sobre machine learning — la interpretación de la Sección 11 no es una decoración opcional, es el objetivo real de usar datos tan antiguos.***